# Preprocessing

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from google.colab import drive
import os

import torch.nn as nn
from torchvision import models

import torch.optim as optim
from tqdm.auto import tqdm

In [ ]:
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
data_dir = '/content/drive/MyDrive/animals'

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size])

train_data.dataset.transform = train_transforms
val_data.dataset.transform = val_transforms

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)

print(f"Classes found: {full_dataset.classes}")
print(f"Training samples: {len(train_data)}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/animals'

# Model Training

In [ ]:
import torch.nn as nn
from torchvision import models

In [ ]:
num_classes = len(full_dataset.classes)

In [ ]:
model = models.resnet18(weights='IMAGENET1K_V1')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 205MB/s]


In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
model.fc = nn.Linear(model.fc.in_features, num_classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model loaded and moved to {device}")

Model loaded and moved to cuda


# Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001) # Only train the new head
epochs = 10

In [ ]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Create a progress bar for the training loader
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", unit="batch")

    for inputs, labels in train_bar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update the progress bar suffix with live stats
        train_bar.set_postfix(loss=running_loss/len(train_loader), acc=f"{100.*correct/total:.2f}%")

    # 3. Validation Phase
    model.eval()
    val_correct = 0
    val_total = 0

    # Create a progress bar for the validation loader
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", unit="batch", leave=False)

    with torch.no_grad():
        for inputs, labels in val_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

            val_bar.set_postfix(val_acc=f"{100.*val_correct/val_total:.2f}%")

    print(f"--- Epoch {epoch+1} Summary: Train Acc: {100.*correct/total:.2f}% | Val Acc: {100.*val_correct/val_total:.2f}% ---")

print("Training Complete!")

Epoch 1/10 [Train]:   0%|          | 0/270 [00:00<?, ?batch/s]

Epoch 1/10 [Val]:   0%|          | 0/68 [00:00<?, ?batch/s]

--- Epoch 1 Summary: Train Acc: 51.13% | Val Acc: 78.33% ---


Epoch 2/10 [Train]:   0%|          | 0/270 [00:00<?, ?batch/s]

Epoch 2/10 [Val]:   0%|          | 0/68 [00:00<?, ?batch/s]

...

Epoch 9/10 [Val]:   0%|          | 0/68 [00:00<?, ?batch/s]

--- Epoch 9 Summary: Train Acc: 96.67% | Val Acc: 88.70% ---


Epoch 10/10 [Train]:   0%|          | 0/270 [00:00<?, ?batch/s]

Epoch 10/10 [Val]:   0%|          | 0/68 [00:00<?, ?batch/s]

--- Epoch 10 Summary: Train Acc: 96.99% | Val Acc: 89.54% ---
Training Complete!


Saving the Model

In [ ]:
from google.colab import drive
import torch
import os

# Define save path in Drive
save_dir = '/content/drive/MyDrive/models'
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, 'model.pth')

# Save the model
torch.save({
    'epoch': epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, model_path)

print(f"Model saved to {model_path}")

NameError: name 'epochs' is not defined

Testing the model

In [6]:
from google.colab import drive
import torch
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
from PIL import Image

drive.mount('/content/drive')

# ── 1. Recreate the exact same architecture ────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class_names = ['antelope', 'badger', 'bat', 'bear', 'bee', 'beetle', 'bison', 'boar', 'butterfly', 'cat', 'caterpillar', 'chimpanzee', 'cockroach', 'cow', 'coyote', 'crab', 'crow', 'deer', 'dog', 'dolphin', 'donkey', 'dragonfly', 'duck', 'eagle', 'elephant', 'flamingo', 'fly', 'fox', 'goat', 'goldfish', 'goose', 'gorilla', 'grasshopper', 'hamster', 'hare', 'hedgehog', 'hippopotamus', 'hornbill', 'horse', 'hummingbird', 'hyena', 'jellyfish', 'kangaroo', 'koala', 'ladybugs', 'leopard', 'lion', 'lizard', 'lobster', 'mosquito', 'moth', 'mouse', 'octopus', 'okapi', 'orangutan', 'otter', 'owl', 'ox', 'oyster', 'panda', 'parrot', 'pelecaniformes', 'penguin', 'pig', 'pigeon', 'porcupine', 'possum', 'raccoon', 'rat', 'reindeer', 'rhinoceros', 'sandpiper', 'seahorse', 'seal', 'shark', 'sheep', 'snake', 'sparrow', 'squid', 'squirrel', 'starfish', 'swan', 'tiger', 'turkey', 'turtle', 'whale', 'wolf', 'wombat', 'woodpecker', 'zebra']  # ⚠️ Replace with your actual classes
                                            # Check what was printed: "Classes found: [...]"
num_classes = len(class_names)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)  # must match training

# ── 2. Load your saved checkpoint ─────────────────────────────────────────────
model_path = '/content/drive/MyDrive/models/model.pth'
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

model.to(device)
model.eval()
print("✅ Model loaded successfully!")

# ── 3. Same transforms as your val_transforms ─────────────────────────────────
predict_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ── 4. Prediction function ─────────────────────────────────────────────────────
def predict(image_path):
    img = Image.open(image_path).convert('RGB')
    tensor = predict_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        predicted_idx = probs.argmax().item()
        confidence = probs[predicted_idx].item()

    print(f"\n🖼️  Image      : {image_path}")
    print(f"🏆 Prediction : {class_names[predicted_idx]}")
    print(f"📊 Confidence : {confidence*100:.2f}%")

    return class_names[predicted_idx], confidence

# ── 5. Run predictions ─────────────────────────────────────────────────────────

# Multiple images
image_paths = [
    '/content/drive/MyDrive/animals/1Tests/shark.jpg',
    '/content/drive/MyDrive/animals/1Tests/bear.jpg'
]
for path in image_paths:
    predict(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model loaded successfully!

🖼️  Image      : /content/drive/MyDrive/animals/1Tests/shark.jpg
🏆 Prediction : shark
📊 Confidence : 97.11%

🖼️  Image      : /content/drive/MyDrive/animals/1Tests/bear.jpg
🏆 Prediction : bear
📊 Confidence : 99.99%
